# P3: Embeddings + HNSW

MinHash-LSH only finds duplicates that share many words. Here every question is turned into a dense vector (an *embedding*) that captures its meaning, so paraphrases like "How do I lose weight?" / "What is the best way to slim down?" end up close to each other even with almost no words in common.

Comparing all ~537k questions with each other would mean billions of pairs, which is too many. Instead we use an approximate nearest-neighbour index (HNSW) to retrieve, for each question, only its most similar questions.

**Pipeline:** embed (cached) -> build HNSW index -> tune `efSearch` against exact search -> search all questions -> tune the similarity threshold on **dev** -> save `emb_candidates.parquet` -> final evaluation on **test** vs LSH.

## Configuration
All settings in one place. `M`, `K` and `EF` control the HNSW search, and the thresholds are the similarity cut-offs tested on dev.

In [1]:
import os
import sys
import time
import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
sys.path.insert(0, "..")
from src.evaluate import evaluate, recall_by_jaccard_bin
from src.embeddings import (load_or_compute_embeddings, build_hnsw, tune_ef_search,
                            search_candidates, tune_threshold)


DATA = '../data/processed'
EMB_PATH = f'{DATA}/embeddings.npy'
QID_PATH = f'{DATA}/embeddings_qids.npy'
CAND_PATH = f'{DATA}/emb_candidates.parquet'

MODEL_NAME = 'all-MiniLM-L6-v2' # try alternatives: 'all-mpnet-base-v2'
M = 32 # HNSW graph degree, can be tuned
K = 20 # neighbours retrieved per question, can be tuned
EF = 64 # efSearch, chosen during tuning below
MIN_THRESHOLD = 0.60 # keep everything above this, threshold is tuned on dev
THRESHOLDS = [0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]

W1114 16:51:21.209000 84402 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## 1. Load questions and compute embeddings
Each question is encoded by `all-MiniLM-L6-v2`, a small pretrained sentence-transformer, into a 384-dimensional vector. Vectors are L2-normalised, so the inner product of two vectors equals their cosine similarity (1 means same meaning).

Encoding takes a few minutes, so the result is saved to `embeddings.npy` together with the qid order. On later runs it is loaded from disk, as long as the qids still match.

In [2]:
df = pd.read_parquet(f'{DATA}/questions.parquet')
texts = df['text_norm'].fillna('').tolist()
qids = df['qid'].tolist()

embeddings = load_or_compute_embeddings(texts, qids,
                                        EMB_PATH, QID_PATH, MODEL_NAME)
print(embeddings.shape)

Loaded cached embeddings
(537174, 384)


## 2. Build the HNSW index

Hierarchical Navigable Small World (HNSW) is a layered graph: each vector is linked to about `M` similar vectors. Upper layers have few nodes and long links, lower layers have all nodes and short links. A search starts at the top, greedily moves towards the query, and goes down layer by layer. This takes a few hundred comparisons instead of one per question.

In [3]:
index_hnsw = build_hnsw(embeddings, M)

Index build time (M=32): 46.7s


## 3. Tune `efSearch`: recall vs speed

`efSearch` is how many candidates HNSW keeps while searching. Higher values give more accurate but slower results. To pick it, we compare HNSW against **exact** search (`IndexFlatIP`, brute force) on 2,000 random questions and measure **recall@k**: the share of the true top-k neighbours that HNSW also finds. No labels are used here. A small warm-up search runs first so the first timing isn't inflated.

In [4]:
df_tuning = tune_ef_search(index_hnsw, embeddings, K)
df_tuning

,efSearch,recall@k,time_s,speedup_vs_exact
0,16,0.956775,0.180967,6.278201
1,32,0.971975,0.141523,8.027991
2,64,0.979700,0.256767,4.424814
3,128,0.986675,0.421140,2.697788
4,256,0.991675,0.753967,1.506891


## 4. Search all questions and build candidate pairs

Every question retrieves its `K` nearest neighbours with the chosen `efSearch`. Each (question, neighbour) becomes a candidate pair with its cosine score. `build_candidates` removes self-matches and invalid results, orders each pair as `qid_a < qid_b`, and drops A,B / B,A duplicates, giving the same format as LSH file. We keep everything above 0.60 so the threshold can be tuned afterwards without searching again.

In [5]:
cands_all = search_candidates(index_hnsw, embeddings, qids, K, EF, MIN_THRESHOLD)
print(len(cands_all), 'candidate pairs with score >=', MIN_THRESHOLD)

Search time (efSearch=64): 59.0s
4886080 candidate pairs with score >= 0.6


## 5. Tune the similarity threshold on dev and save candidates 

A higher threshold gives fewer, more precise pairs but lower recall. We evaluate several thresholds with the `evaluate` function on the **dev** split and keep the one with the best F1. The resulting pairs are saved as `emb_candidates.parquet`.

In [6]:
pairs = pd.read_parquet(f'{DATA}/pairs.parquet')

df_thr, best_t = tune_threshold(cands_all, pairs, df, THRESHOLDS, split='dev')
display(df_thr[['threshold', 'n_candidates', 'recall', 'precision', 'f1', 'recall_implied']])
print('Best threshold on dev (F1):', best_t)

df_candidates = cands_all[cands_all['score'] >= best_t].reset_index(drop=True)
df_candidates.to_parquet(CAND_PATH, index=False)
print(len(df_candidates), 'pairs saved to', CAND_PATH)

,threshold,n_candidates,recall,precision,f1,recall_implied
0,0.60,4886080,0.890037,0.530639,0.664879,0.713889
1,0.65,3516782,0.884076,0.557181,0.683557,0.706200
2,0.70,2317205,0.867823,0.591020,0.703161,0.686162
3,0.75,1417332,0.828573,0.632087,0.717114,0.641841
4,0.80,798751,0.747038,0.678597,0.711175,0.561135
5,0.85,408967,0.614028,0.732548,0.668072,0.442414
6,0.90,180191,0.429257,0.794608,0.557400,0.295993
7,0.95,55638,0.210248,0.865075,0.338280,0.139960


Best threshold on dev (F1): 0.75
1417332 pairs saved to ../data/processed/emb_candidates.parquet


## 6. Final evaluation on test vs LSH

Run once, after every choice (efSearch, threshold) has been fixed on dev. Both systems are evaluated on the **test** split. The second table splits true duplicates by word overlap (Jaccard): it shows that embeddings also recover duplicates with little word overlap, which LSH by design cannot.

In [7]:
lsh = pd.read_parquet(f'{DATA}/lsh_candidates.parquet')

test_res = pd.DataFrame([
    {'system': 'lsh', **evaluate(lsh, pairs, df, split='test')},
    {'system': f'emb (t={best_t}, ef={EF})', **evaluate(df_candidates, pairs, df, split='test')},
])
display(test_res)

bins = pd.concat({
    'lsh': recall_by_jaccard_bin(lsh, pairs, df, split='test')['recall'],
    'emb': recall_by_jaccard_bin(df_candidates, pairs, df, split='test')['recall'],
}, axis=1)
bins

,system,n_candidates,fraction_of_all_pairs,labelled_candidates,recall,precision,f1,recall_implied
0,lsh,26906047,0.000186,17089,0.286274,0.499912,0.364066,0.189243
1,"emb (t=0.75, ef=64)",1417332,0.000010,37946,0.805610,0.633558,0.709300,0.613130


,lsh,emb
bin,,
"(-0.001, 0.2]",0.000000,0.490095
"(0.2, 0.4]",0.018804,0.696026
"(0.4, 0.6]",0.234355,0.902285
"(0.6, 0.8]",0.836091,0.976251
"(0.8, 1.0]",0.999499,0.994489
